# 水印模型评估（效用 / 保真度）

把处理（水印）后的实体嵌入写回 RotatE 模型，重跑链路预测指标，
对比水印前后的 **GMR / HMR / AMR / Hits@10**，确认水印没有明显破坏嵌入效用。

- 输入：`dataprocess.ipynb` 生成的 `model.pt` / `train_triples.pt` / `test_triples.pt`，
  以及水印后的实体嵌入文件（默认 `../datasets/<dataset>/watermarked_embeddings.pt`）。
- 期望格式：`(num_entities, 4096)`、`torch.complex64`；实数版 `(num_entities, 8192)` 也会自动转换。
- 关系嵌入与交互函数保持不变，只替换实体嵌入。

In [ ]:
import os

import torch

import pandas as pd
from pykeen.evaluation import RankBasedEvaluator

DATASETS = ['ali', 'lastfm', 'mind']

# 指标 key（side='both', rank_type='realistic'）
METRICS = {
    'GMR': 'geometric_mean_rank',   # ↓ 越低越好
    'HMR': 'harmonic_mean_rank',    # ↓ 越低越好
    'AMR': 'arithmetic_mean_rank',  # ↓ 越低越好
    'Hits@10': 'hits_at_10',        # ↑ 越高越好
}

# 水印后的实体嵌入文件（由你的水印管线输出）。不存在时回退到原始 embeddings.pt。
WATERMARKED_TEMPLATE = '../datasets/{dataset}/watermarked_embeddings.pt'

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def load_model_and_triples(dataset):
    model = torch.load(f'../datasets/{dataset}/model.pt', map_location=DEVICE, weights_only=False)
    train_triples = torch.load(f'../datasets/{dataset}/train_triples.pt')
    test_triples = torch.load(f'../datasets/{dataset}/test_triples.pt')
    return model, train_triples, test_triples


def evaluate_model(model, train_triples, test_triples):
    evaluator = RankBasedEvaluator(filtered=True)
    results = evaluator.evaluate(
        model=model,
        mapped_triples=test_triples,
        additional_filter_triples=[train_triples, test_triples],
    )
    return {name: results.get_metric(key) for name, key in METRICS.items()}


def inject_entity_embeddings(model, embeddings):
    """把处理后的实体嵌入写回模型（关系嵌入不变）。"""
    ent_rep = model.entity_representations[0]
    storage = ent_rep._embeddings.weight  # (num_entities, 2*dim) float32，RotatE 复数存储

    if not embeddings.is_complex():
        # 实数输入 (num_entities, 2*dim) -> 复数 (num_entities, dim)
        embeddings = torch.view_as_complex(embeddings.reshape(storage.shape[0], -1, 2))
    embeddings = embeddings.to(torch.complex64)  # storage 是 float32，对应 complex64

    assert tuple(embeddings.shape) == (storage.shape[0], storage.shape[1] // 2), embeddings.shape
    storage.data.copy_(torch.view_as_real(embeddings).reshape(storage.shape))

## 运行评估

In [ ]:
rows = []
for dataset in DATASETS:
    model, train_triples, test_triples = load_model_and_triples(dataset)
    original = evaluate_model(model, train_triples, test_triples)

    wm_path = WATERMARKED_TEMPLATE.format(dataset=dataset)
    if os.path.exists(wm_path):
        watermarked = torch.load(wm_path)
    else:
        print(f'[warn] {wm_path} 不存在，回退到原始 embeddings.pt（结果将等于 original）')
        watermarked = torch.load(f'../datasets/{dataset}/embeddings.pt')

    inject_entity_embeddings(model, watermarked)
    watermarked_metrics = evaluate_model(model, train_triples, test_triples)

    print(f'{dataset}:')
    for name in METRICS:
        print(f'  {name:8s} original={original[name]:.4f}  watermarked={watermarked_metrics[name]:.4f}')
    print()

    for name in METRICS:
        rows.append({'dataset': dataset, 'model': 'original', 'metric': name, 'value': original[name]})
        rows.append({'dataset': dataset, 'model': 'watermarked', 'metric': name, 'value': watermarked_metrics[name]})

comparison = pd.DataFrame(rows).pivot_table(index='dataset', columns=['model', 'metric'], values='value')
comparison